The following [URL](http://api.worldbank.org/v2/country/US/indicator/SP.POP.TOTL?per_page=100) pulls the total population in the USA over time in XML format
`http://api.worldbank.org/v2/country/US/indicator/SP.POP.TOTL?per_page=100`

What year was the largest growth since the year before?

In [1]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
response = requests.get("http://api.worldbank.org/v2/country/US/indicator/SP.POP.TOTL?per_page=100")
print(response.status_code)
if response.status_code == 200:
    text = response.text

200


In [3]:
root = ET.fromstring(text[3:])

In [4]:
res = []
for datapoint in root.findall("./{http://www.worldbank.org}data"):
    year = datapoint.find("{http://www.worldbank.org}date").text
    population = datapoint.find("{http://www.worldbank.org}value").text
    res.append({"year":year, "population":population})

In [5]:
df = pd.DataFrame(res)

In [6]:
df.dtypes

year          object
population    object
dtype: object

In [7]:
df['population'] = pd.to_numeric(df['population'])

In [8]:
df['year'] = pd.to_datetime(df['year'])

In [9]:
series = df.set_index('year')['population'].sort_index()
series

year
1960-01-01    180671000
1961-01-01    183691000
1962-01-01    186538000
1963-01-01    189242000
1964-01-01    191889000
                ...    
2020-01-01    331577720
2021-01-01    332099760
2022-01-01    334017321
2023-01-01    336806231
2024-01-01    340110988
Name: population, Length: 65, dtype: int64

In [10]:
delta_pop = 100* ( series - series.shift() ) / series.shift()

In [11]:
delta_pop.sort_values(ascending=False).head()

year
1961-01-01    1.671547
1962-01-01    1.549885
1963-01-01    1.449571
1964-01-01    1.398738
1992-01-01    1.396548
Name: population, dtype: float64